# Corticall - Gate 0 v2: frontal-face-present vs face-absent (GO / NO-GO)

Kaggle **GPU T4 + Internet ON**. Recipe: **Run All -> Restart & Clear -> Run All**.

Fixes the AMBIGUOUS Run 1. Pre-registered design (D021 / `notebooks/gate0_v2_stimuli.json`): **FACE** = single frontal face fills the frame; **NONFACE** = no dominant frontal face (people-with-backs/profiles/scenes) - both from the **same** public-domain film *Charade (1963)*, so the only systematic difference is a frontal face (controls film/people/scene/low-level at once). **GO** = right-FFC face>nonface (perm p<=0.025), > V1 and > EBA (specificity), surviving video-only. McLintock landscapes give a PPA place positive-control (reported). ~68 passes, ~7h, HDF5-cached.

## Setup (reused verbatim from `01_setup_test.ipynb`)
### Phase 1 - install TRIBE v2 (+ shadow/numpy guards)

In [ ]:
import sys, subprocess, shutil, importlib
print('Python:', sys.version)
SRC = '/kaggle/working/tribev2_src'
shutil.rmtree('/kaggle/working/tribev2', ignore_errors=True)
shutil.rmtree(SRC, ignore_errors=True)
for _m in [m for m in list(sys.modules) if m == 'tribev2' or m.startswith('tribev2.')]:
    sys.modules.pop(_m, None)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=False)
subprocess.run(f'git clone --depth 1 https://github.com/facebookresearch/tribev2.git {SRC}',
               shell=True, check=False)
TRIBEV2_SHA = subprocess.run(['git', '-C', SRC, 'rev-parse', 'HEAD'],
                             capture_output=True, text=True).stdout.strip()
print('tribev2 commit:', TRIBEV2_SHA or '(unknown)')
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', SRC],
                   capture_output=True, text=True)
print('tribev2 install rc =', r.returncode)
if r.returncode != 0:
    print('--- STDERR (tail) ---'); print(r.stderr[-3000:])
    print('>>> If this mentions requires-python / neuralset>=3.12, Kaggle is on Python <3.12 (G016).')
# exca >= 0.5.26 removed exca.steps.base.NoValue, which neuralset 0.0.2 still references:
# 0.5.20 / 0.5.25 import fine, 0.5.26 (2026-06-03) .. 0.5.29 (2026-07-28) raise
# AttributeError in neuralset/events/study.py. tribev2 pins neuralset==0.0.2 but nothing
# pins exca, so an unpinned resolve can kill the run at minute zero. Pin it.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'exca==0.5.25'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--force-reinstall', '--no-deps',
                '--no-cache-dir', '-q', 'numpy==2.2.6'], check=False)
if SRC not in sys.path:
    sys.path.insert(0, SRC)
importlib.invalidate_caches()
print('install step done - inspect above for resolver/version errors')
import numpy as _np
print('numpy:', _np.__version__)
import tribev2
where = getattr(tribev2, '__file__', None) or list(getattr(tribev2, '__path__', []))
print('tribev2 resolves to:', where)
assert where and 'tribev2_src' in str(where), (
    f'SHADOWED: tribev2 resolved to {where}, not {SRC}. Restart (Factory reset) and re-run.')
importlib.import_module('tribev2.demo_utils')
import exca as _exca
EXCA_VERSION = getattr(_exca, '__version__', '?')
print('exca:', EXCA_VERSION)
print('OK: tribev2.demo_utils imports.')

### Clone Corticall (tribe-bench) - brings `tribe_tools.roi_stats` etc.

In [ ]:
import os, sys, subprocess, glob
from pathlib import Path
TB_PATH = None
subprocess.run('git clone --depth 1 https://github.com/codesbydevesh/tribe-bench.git /kaggle/working/tribe-bench',
               shell=True, check=False)
if Path('/kaggle/working/tribe-bench/tribe_tools/model.py').is_file():
    TB_PATH = '/kaggle/working/tribe-bench'
if TB_PATH is None:
    for cand in glob.glob('/kaggle/input/*') + glob.glob('/kaggle/input/*/*'):
        if Path(cand, 'tribe_tools', 'model.py').is_file():
            TB_PATH = cand; break
if TB_PATH:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', TB_PATH], check=False)
    if TB_PATH not in sys.path:
        sys.path.insert(0, TB_PATH)
    print('tribe-bench found at:', TB_PATH)
else:
    print('tribe-bench NOT available - check the clone error above (repo should be public).')

### Phase 2 - environment

In [ ]:
import torch
print('torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f'  GPU {i}: {p.name}, {p.total_memory/1e9:.1f} GB')
else:
    print('NO GPU - enable a T4 accelerator in Kaggle Settings, then re-run.')
import shutil
print('ffmpeg on PATH:', bool(shutil.which('ffmpeg')))
print('uvx on PATH   :', bool(shutil.which('uvx')), '(required for WhisperX ASR)')

### Phase 3 - HuggingFace login (gated LLaMA-3.2)

In [ ]:
import os
hf_ok = False
try:
    from huggingface_hub import login
    token = os.environ.get('HF_TOKEN', '')
    if not token:
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret('HF_TOKEN')
        except Exception:
            token = ''
    if token:
        login(token=token); os.environ['HF_TOKEN'] = token; hf_ok = True
        print('HF login OK')
    else:
        print('No HF_TOKEN. Set a Kaggle secret HF_TOKEN. The text (LLaMA) extractor will fail without it.')
except Exception as e:
    print('HF login error:', type(e).__name__, e)

### Phase 4 - import the wrapper

In [ ]:
WRAPPER_OK = False
try:
    from tribe_tools.model import load_model, predict_single, MODALITY_MASKS
    WRAPPER_OK = True
    print('tribe_tools.model imports: OK'); print('MODALITY_MASKS:', MODALITY_MASKS)
except Exception as e:
    print('WRAPPER IMPORT FAILED:', type(e).__name__, e)
for _m in ['tribe_tools.atlas', 'tribe_tools.cache', 'tribe_tools.viz',
           'tribe_tools.roi_stats', 'neurocheck.claims']:
    try:
        __import__(_m); print('  optional import OK:', _m)
    except Exception as e:
        print('  optional import skipped:', _m, '->', type(e).__name__, e)

## Gate 0 v2 prep - download films, cut clips from the frozen manifest, atlas pre-flight

In [ ]:
# ===== GATE 0 v2 - PREP: download 2 PD films, cut clips from the FROZEN manifest =====
import json, subprocess
from pathlib import Path
import numpy as np

CACHE = Path('/kaggle/working/cache'); CACHE.mkdir(parents=True, exist_ok=True)
CLIPS = Path('/kaggle/working/clips'); CLIPS.mkdir(parents=True, exist_ok=True)
# D023(g): v3 supersedes v2. v2 was measured to have a motion confound (p=0.0008) and clips
# averaging 1.5-2.2 shot changes each; v3 is sustained-single-shot and motion-matched.
# v2 is kept on disk unchanged as the historical record.
_MAN = '/kaggle/working/tribe-bench/notebooks/gate0_v3_stimuli.json'
M = json.load(open(_MAN))
print('stimulus manifest:', _MAN.split('/')[-1], '|', M['design'])
DUR = M['clip_dur_s']

def fetch(url, out):
    out = Path(out)
    if not out.exists() or out.stat().st_size == 0:
        subprocess.run(['bash', '-lc', f'curl -L --fail -o "{out}" "{url}"'], check=False)
    return out

charade = fetch(M['primary_source']['url'], CACHE / 'charade.mp4')
mcl = fetch(M['confirmatory_scene_source']['url'], CACHE / 'mclintock.mp4')
print('charade:', charade.stat().st_size // 1_000_000, 'MB | mclintock:', mcl.stat().st_size // 1_000_000, 'MB')

def cut(src, start, name):
    out = CLIPS / f'{name}.mp4'
    subprocess.run(['ffmpeg', '-y', '-ss', str(start), '-t', str(DUR), '-i', str(src),
                    '-vf', 'scale=-2:480', '-c:v', 'libx264', '-preset', 'veryfast', '-crf', '23',
                    '-c:a', 'aac', str(out)], check=False, capture_output=True)
    return out

FACE = [cut(charade, s, f'FACE_{i:02d}') for i, s in enumerate(M['face_starts_s'])]
NONFACE = [cut(charade, s, f'NONFACE_{i:02d}') for i, s in enumerate(M['nonface_starts_s'])]
SCENE = [cut(mcl, s, f'SCENE_{i:02d}') for i, s in enumerate(M['confirmatory_scene_source']['scene_starts_s'])]
print('cut:', len(FACE), 'face,', len(NONFACE), 'nonface,', len(SCENE), 'scene(confirm)')

import matplotlib.pyplot as plt
def montage(clips, title, fname):
    n = len(clips); cols = 5; rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(16, 2.6 * rows))
    axf = np.atleast_1d(axes).ravel()
    for ax, p in zip(axf, clips):
        th = CLIPS / f'thumb_{p.stem}.png'
        subprocess.run(['ffmpeg', '-y', '-ss', '0', '-i', str(p), '-frames:v', '1', str(th)],
                       check=False, capture_output=True)
        try: ax.imshow(plt.imread(th))
        except Exception: pass
        ax.set_title(p.stem, fontsize=8); ax.axis('off')
    for ax in axf[n:]: ax.axis('off')
    plt.suptitle(title); plt.tight_layout(); plt.savefig(f'/kaggle/working/{fname}'); plt.show()
montage(FACE, 'FACE (frontal face fills frame)', 'm_face.png')
montage(NONFACE, 'NONFACE (no dominant frontal face)', 'm_nonface.png')
print('>>> sanity (FRAME 0 - the frame V-JEPA2 over-weights): top = clear frontal faces; bottom = people, no dominant frontal face.')

# atlas pre-flight: FFC(face), EBA proxy(body), V1, A1, PPA
from tribe_tools import atlas
from tribe_tools.cache import get_cache
regions = set(atlas.list_regions())
def verts(names, hemi='both'):
    if isinstance(names, str): names = [names]
    keep = [n for n in names if n in regions]
    return np.concatenate([atlas.get_vertices(n, hemi=hemi) for n in keep]) if keep else np.array([], dtype=int)
FFCr = verts('FFC', 'right')
EBA = verts(['LO2', 'LO3', 'V4t', 'FST', 'PH'], 'right')      # body region (replaces the 12-vtx LO2)
V1v = verts('V1'); A1v = verts('A1')
PPA = verts(['PHA1', 'PHA2', 'PHA3', 'VMV1', 'VMV2', 'VMV3'])
for nm, v in [('FFCr', FFCr), ('EBA', EBA), ('V1', V1v), ('A1', A1v), ('PPA', PPA)]:
    print(f'  {nm}: {len(v)} vertices'); assert len(v) > 0, nm
cache = get_cache(CACHE / 'gate0v2')
print('PREP OK. ~68 passes x ~6 min ~= 7h (HDF5-cached, resumes if the session dies).')

## Load the model (reused; uses CACHE from prep)

In [ ]:
import time
tribe = None; load_time = None
if WRAPPER_OK:
    try:
        t0 = time.time()
        tribe = load_model(device='cuda', cache_folder=CACHE)
        load_time = time.time() - t0
        print(f'Model loaded in {load_time:.1f}s')
    except Exception as e:
        import traceback; traceback.print_exc()
        print('MODEL LOAD FAILED:', type(e).__name__, e)
else:
    print('Skipped: wrapper not importable.')

## Pre-flight - loosen the ASR sentence-alignment guard (10s clips are too short for the 5% default)

In [ ]:
# ===== PRE-FLIGHT: ASR sentence-alignment tolerance (G019) + text-pipeline instrumentation =====
# demo_utils.get_audio_and_text_events() builds AddSentenceToWords(max_unmatched_ratio=0.05) at CALL
# time and resolves the name from the demo_utils module global, so rebinding that global takes effect.
# (Class default is 0.0; 0.05 is the call-site kwarg.)
#
# WHY THIS IS NOT COSMETIC - corrects D022's C1/C2, which were FALSE: a word that fails alignment
# gets sentence="" (neuralset text.py:186), AddContextToWords then gives it context="" (text.py:271-274),
# and RemoveMissing DELETES the row (basic.py:52-54); its TRs receive exact zeros from the text
# extractor (extractors/base.py:250-262, 302-305). The ratio is the ONLY cap on a destructive step,
# so widening it does not "lose nothing but the guard" and the loss is not zero-mean noise.
# Hence 0.15, not 0.99: admits the observed 1/9 = 0.111 that killed FACE_08, rejects 2/9 = 0.222.
# The check is the LAST statement of _run (text.py:215-224), after annotations are written, so any
# clip that passed at 0.05 is byte-identical at 0.15. Nothing here is exca-cached (the transforms run
# with infra=None, transforms/base.py:156-158), so no feature cache is affected either way.
# Must run before BOTH pass loops: predict_single builds events (running ASR + alignment) before
# applying features_to_mask, so the video-only pass goes through this too.
# CAVEAT, pre-existing and NOT fixed here: a clip with ZERO ASR words returns at text.py:174-176
# without raising at any tolerance, and silently loses the whole text modality. TEXTLOG will show it.
import tribev2.demo_utils as _du
import pandas as _pd

ASW_RATIO = 0.15
_real_asw = getattr(_du, '_gate0v2_real_asw', _du.AddSentenceToWords)
_du._gate0v2_real_asw = _real_asw          # key off the REAL class so re-running this cell re-applies
def _lenient_asw(**kw):
    kw['max_unmatched_ratio'] = ASW_RATIO
    return _real_asw(**kw)
_du.AddSentenceToWords = _lenient_asw
_du._gate0v2_lenient_asw = True

# Instrumentation. RemoveMissing is where words actually die, so count them there, per clip.
# Recorded, never asserted - an assert at hour 7 destroys the run.
TEXTLOG = {}
CURRENT_CLIP = {'name': '?'}
from neuralset.events.transforms.basic import RemoveMissing as _RM
_real_rm = getattr(_du, '_gate0v2_real_rm', _RM._run)
_du._gate0v2_real_rm = _real_rm

def _count_words(df):
    try:
        return int((df['type'].astype(str) == 'Word').sum())
    except Exception:
        return -1

def _rm_logged(self, events):
    before = _count_words(events)
    unmatched = -1
    try:
        w = events[events['type'].astype(str) == 'Word']
        if len(w) and 'sentence' in w.columns:
            # same predicate as the guard at text.py:218
            unmatched = int(sum(not s or not isinstance(s, str) for s in w['sentence']))
    except Exception:
        pass
    out = _real_rm(self, events)
    after = _count_words(out)
    rec = TEXTLOG.setdefault(CURRENT_CLIP['name'], {})
    rec.update(words_before=before, words_after=after,
               words_deleted=(before - after if before >= 0 and after >= 0 else None),
               unmatched=unmatched,
               unmatched_ratio=(round(unmatched / before, 4) if before > 0 and unmatched >= 0 else None))
    return out

_RM._run = _rm_logged

print(f'AddSentenceToWords.max_unmatched_ratio -> {_du.AddSentenceToWords().max_unmatched_ratio}'
      '  (call site was 0.05, class default 0.0)')
print('RemoveMissing instrumented -> TEXTLOG carries per-clip word counts and deletions.')


## FULL passes

In [ ]:
# ===== GATE 0 v2 - FULL passes: SCENE(confirm) + FACE + NONFACE =====
# SCENE runs FIRST so the PPA place positive control (Run 1: dz=+2.03, U=16/16) lands in the first
# ~50 min. If the Kaggle session dies mid-run we still hold a working positive control.
# FN/NN/SN in the analysis derive from the clip LISTS, not from dict order, so this is safe.
from tribe_tools.model import predict_single
import tribev2.demo_utils as _du
assert getattr(_du, '_gate0v2_lenient_asw', False), \
    'cell 16 pre-flight not applied - run cell 16 before the pass loops'

FAILED = {}
def run(v, mask):
    tag = 'full' if not mask else 'vid'
    CURRENT_CLIP['name'] = v.name                     # so cell 16's TEXTLOG attributes to this clip
    # size is IN THE KEY: clip files are named positionally (FACE_07) and ffmpeg runs with -y, so a
    # mid-session re-cut would otherwise serve the OLD clip's predictions under the NEW clip's label.
    key = f"{v.resolve()}_{v.stat().st_size}_{tag}"
    hit = cache.load(key)
    if hit is not None:
        print('  cache', v.name); return hit
    try:
        preds, _ = predict_single(tribe, v, features_to_mask=mask)
    except Exception as e:
        # a single bad clip must not destroy a 7h run - record it, the analysis cell
        # drops it from both conditions and prints what was lost (never silently)
        FAILED[f'{v.name}:{tag}'] = f'{type(e).__name__}: {e}'
        print('  FAILED', v.name, tag, '->', type(e).__name__, e); return None
    rec = TEXTLOG.setdefault(v.name, {})
    rec[f'n_kept_{tag}'] = int(preds.shape[0])
    if 'asr_words_tsv' not in rec:                     # whisperx writes a .tsv beside the clip - free ground truth
        try:
            hits = sorted(v.parent.glob(v.stem + '*.tsv'))
            if hits:
                rec['asr_words_tsv'] = int(len(_pd.read_csv(hits[0], sep='\t')))
        except Exception:
            pass
    cache.save(key, preds, metadata={'clip': v.name, 'mask': tag})
    print('  ran', v.name, preds.shape); return preds

assert tribe is not None, 'model not loaded'
full = {v.name: p for v in SCENE + FACE + NONFACE if (p := run(v, None)) is not None}
print('FULL done:', len(full), '| failed:', len(FAILED))


## VIDEO-ONLY passes (G4 control)

In [ ]:
# ===== GATE 0 v2 - VIDEO-ONLY passes (G4: rules out a speech/audio artifact) =====
vid = {v.name: p for v in FACE + NONFACE if (p := run(v, ['audio', 'text'])) is not None}
print('VIDEO-ONLY done:', len(vid), '| failed:', len(FAILED))

## Analysis + pre-registered verdict

In [ ]:
# ===== GATE 0 v2 - analysis + PRE-REGISTERED verdict (D021, amended by D023) =====
import json, math
from pathlib import Path
import numpy as np
try:
    from tribe_tools.roi_stats import spatial_z, u_statistic, perm_p
except Exception:
    def spatial_z(preds, v):
        g = preds.mean(0) if preds.ndim == 2 else np.asarray(preds); sd = g.std()
        return 0.0 if sd == 0 else float((g[v].mean() - g.mean()) / sd)
    def u_statistic(a, b):
        u = 0.0
        for x in a:
            for y in b: u += 1.0 if x > y else (0.5 if x == y else 0.0)
        return u
    def perm_p(a, b, n_perm=10000, seed=0):
        vals = np.array(list(a) + list(b), float); n = len(a); N = len(vals)
        uo = u_statistic(vals[:n], vals[n:]); rng = np.random.default_rng(seed); ge = 0
        for _ in range(n_perm):
            p = rng.permutation(N)
            if u_statistic(vals[p[:n]], vals[p[n:]]) >= uo - 1e-9: ge += 1
        return (ge + 1) / (n_perm + 1)

N_PERM, SEED_P, SEED_D = 10000, 0, 1

def mc_delta_thr(a, b, n_perm=N_PERM, seed=SEED_D, pct=95):
    vals = np.array(list(a) + list(b), float); n = len(a); N = len(vals); rng = np.random.default_rng(seed)
    ds = np.array([(lambda p: vals[p[:n]].mean() - vals[p[n:]].mean())(rng.permutation(N)) for _ in range(n_perm)])
    return float(np.percentile(ds, pct))

# only clips that survived every pass they appear in enter the contrast - n is reported, never padded
def usable(clips, *passes): return [p.name for p in clips if all(p.name in q for q in passes)]
FN = usable(FACE, full, vid); NN = usable(NONFACE, full, vid); SN = usable(SCENE, full)
if FAILED:
    print('DROPPED', len(FAILED), 'passes:')
    for k, msg in FAILED.items(): print('  ', k, '->', msg)
print(f'n used: face={len(FN)}/{len(FACE)}  nonface={len(NN)}/{len(NONFACE)}  scene={len(SN)}/{len(SCENE)}')
# attrition is only harmless if it is SYMMETRIC - asymmetric drops bias the contrast itself
drop_face, drop_nonface = len(FACE) - len(FN), len(NONFACE) - len(NN)
print(f'attrition: face={drop_face} nonface={drop_nonface} imbalance={abs(drop_face - drop_nonface)}')
# PRINT, never assert: an assert here would destroy 7h of finished compute (D023(d))
underpowered = min(len(FN), len(NN)) < 12
imbalanced = abs(drop_face - drop_nonface) > 2
if underpowered: print('!! WARNING: fewer than 12 clips survived in a condition - verdict is UNDERPOWERED')
if imbalanced: print('!! WARNING: attrition imbalance > 2 clips - contrast may be biased, read TEXTLOG')

def zvals(passes, v, names): return [spatial_z(passes[n], v) for n in names]
def report(passes, v, A, B, name):
    a = zvals(passes, v, A); b = zvals(passes, v, B)
    d = float(np.mean(a) - np.mean(b)); U = u_statistic(a, b); p = perm_p(a, b, N_PERM, SEED_P)
    print(f'{name:12} d={d:+.3f}  U={U:.0f}/{len(a)*len(b)}  p={p:.4f}')
    return dict(name=name, d=d, U=float(U), p=float(p), a=a, b=b)

print('=== FULL: FACE vs NONFACE ===')
FFC = report(full, FFCr, FN, NN, 'FFCr(FFA)'); V1 = report(full, V1v, FN, NN, 'V1')
EB = report(full, EBA, FN, NN, 'EBA(body)'); A1 = report(full, A1v, FN, NN, 'A1')
print('=== VIDEO-ONLY (G4) ==='); FFCv = report(vid, FFCr, FN, NN, 'FFCr vid')
print('=== CONFIRM: SCENE vs FACE in PPA (place control) ==='); PP = report(full, PPA, SN, FN, 'PPA scene>face')

G1 = FFC['p'] <= 0.025
G2 = FFC['d'] > mc_delta_thr(FFC['a'], FFC['b'])
G3 = (FFC['d'] > V1['d']) and (FFC['d'] > EB['d'])
G4 = (FFCv['d'] > 0) and (FFCv['p'] <= 0.05)
# D023(a): NO-GO is restricted to the two SIGN conditions. G3 remains a required conjunct of GO, but a
# non-significant ROI-ORDERING reversal no longer ends the project - it routes to AMBIGUOUS/diagnose.
# Rationale: G3 compares two point estimates with no null over ROIs of 58 / 116 / 523 vertices, and the
# 58-vertex FFC mean is structurally the noisiest of the three. Run 1's OWN observed pattern (the body
# ROI out-responding FFC) would be converted from AMBIGUOUS into NO-GO by the un-amended rule.
if all([G1, G2, G3, G4]):
    verdict = 'GO -> ROADMAP Phase 1'
elif FFC['d'] <= 0 or FFCv['d'] <= 0:
    verdict = 'NO-GO -> stop; D017 static-resource fallback'
else:
    verdict = 'AMBIGUOUS -> diagnose (finer curation / stock-video key), do not build Phase 1'

print(f"\nGATES  G1(sig)={G1}  G2(mag)={G2}  G3(spec: FFC>V1 & FFC>EBA)={G3}  G4(video-only)={G4}")
print('VERDICT:', verdict)
print('confirm PPA scene>face: d=%.3f p=%.4f (expect a strong positive place effect)' % (PP['d'], PP['p']))

# disclosed diagnostic (D023(c)): did the conditions differ in how much speech they carried?
def _words(n):
    r = TEXTLOG.get(n, {})
    for k in ('asr_words_tsv', 'words_before'):
        if isinstance(r.get(k), int) and r[k] >= 0: return r[k]
    return None
wf = [w for w in (_words(n) for n in FN) if w is not None]
wn = [w for w in (_words(n) for n in NN) if w is not None]
speech = None
if len(wf) >= 3 and len(wn) >= 3:
    speech = dict(face_mean=float(np.mean(wf)), nonface_mean=float(np.mean(wn)),
                  p=float(perm_p(wf, wn, N_PERM, SEED_P)), n_face=len(wf), n_nonface=len(wn))
    print('speech asymmetry (disclosure, gates nothing): face_words=%.1f nonface_words=%.1f p=%.4f'
          % (speech['face_mean'], speech['nonface_mean'], speech['p']))
else:
    print('speech asymmetry: not enough word counts logged to report')

# ---- WRITE THE ARTIFACT FIRST. Everything below this point is cosmetic and must not be able to
# ---- destroy a finished 7h run (the old cell plotted BEFORE writing the JSON).
out = dict(verdict=verdict, gates=dict(G1=bool(G1), G2=bool(G2), G3=bool(G3), G4=bool(G4)),
           face_vs_nonface={r['name']: {k: r[k] for k in ('d', 'U', 'p')} for r in [FFC, V1, EB, A1, FFCv]},
           per_clip_z={r['name']: dict(face=r['a'], nonface=r['b']) for r in [FFC, V1, EB, A1, FFCv]},
           confirm_PPA_scene_gt_face={k: PP[k] for k in ('d', 'U', 'p')},
           n_face=len(FN), n_nonface=len(NN), n_scene=len(SN),
           face_clips=FN, nonface_clips=NN, scene_clips=SN,
           attrition=dict(face=drop_face, nonface=drop_nonface,
                          imbalance=abs(drop_face - drop_nonface)),
           underpowered=bool(underpowered), imbalanced=bool(imbalanced),
           dropped=FAILED, text_pipeline=TEXTLOG, speech_asymmetry=speech,
           roi_vertices={'FFCr': int(len(FFCr)), 'V1': int(len(V1v)), 'EBA': int(len(EBA)),
                         'A1': int(len(A1v)), 'PPA': int(len(PPA))},
           stats=dict(n_perm=N_PERM, seed_p=SEED_P, seed_delta=SEED_D, alpha_primary=0.025),
           provenance=dict(tribev2_sha=globals().get('TRIBEV2_SHA'),
                           exca=globals().get('EXCA_VERSION'),
                           asw_max_unmatched_ratio=globals().get('ASW_RATIO')),
           source='Charade (1963), public domain')
# D023(c)/(f): carry the measured stimulus covariates next to the verdict so a reader never sees
# the result without the disclosed confounds. Measured on CPU before the run (see
# notebooks/gate0_v2_preflight.json); if the file is absent the run still proceeds.
try:
    _pf = Path('/kaggle/working/tribe-bench/notebooks/gate0_v2_preflight.json')
    out['stimulus_preflight'] = json.loads(_pf.read_text()) if _pf.is_file() else 'not found'
except Exception as _e:
    out['stimulus_preflight'] = f'unreadable: {_e}'
_nan = [k for k, r in out['face_vs_nonface'].items() if any(not math.isfinite(v) for v in r.values())]
out['nonfinite_stats'] = _nan
if _nan: print('!! WARNING: non-finite statistics in', _nan, '- do NOT trust the verdict')
Path('/kaggle/working/gate0v2_results.json').write_text(json.dumps(out, indent=2))
print('wrote gate0v2_results.json to /kaggle/working - DOWNLOAD IT NOW, before anything else')

try:
    import matplotlib.pyplot as plt
    rows = [FFC, V1, EB, A1]
    plt.figure(figsize=(7, 4)); plt.bar([r['name'] for r in rows], [r['d'] for r in rows])
    plt.axhline(0, color='k', lw=.8); plt.ylabel('delta spatial-z (face - nonface)')
    plt.title('Gate 0 v2: ' + verdict.split(' ->')[0])
    plt.xticks(rotation=15); plt.tight_layout(); plt.savefig('/kaggle/working/gate0v2_contrast.png', dpi=120); plt.show()
    # clip-equal-weighted and shape-safe (np.stack raises if any clip has a different n_kept)
    grand = np.mean([full[n].mean(0) for n in FN], axis=0)
    print('top-k FACE ROIs:', atlas.get_topk_rois(grand, k=10))
except Exception as e:
    print('cosmetic tail failed (results JSON is already written):', type(e).__name__, e)
